# Анализ и предобработка исторических данных о продажах видеоигр (2000–2013)

**Автор:** Зарипова Альмира  
**Дата:** апрель 2026

---

## Цели и задачи проекта

Подготовить и очистить исторические данные о видеоиграх за 2000–2013 гг. для последующего анализа, который ляжет в основу статьи на IT-ресурсе. Цель статьи — привлечь новую аудиторию, показав эволюцию индустрии (особенно RPG-жанра) и связав её с атмосферой «Секретов Темнолесья».

### Задачи

1. Познакомиться с данными — изучить структуру, типы столбцов, выявить пропуски и аномалии
2. Отфильтровать данные по времени — оставить только игры, выпущенные с 2000 по 2013 год включительно
3. Категоризовать оценки — на основе пользовательских и экспертных оценок разделить игры на три категории: высокая, средняя, низкая
4. Выделить топ-7 платформ — определить платформы с наибольшим количеством выпущенных игр за требуемый период
5. Провести предобработку — исправить типы данных, обработать пропуски, подготовить «чистый» срез для анализа

---

## Описание данных

Данные `new_games.csv` содержат информацию о продажах игр разных жанров и платформ, а также пользовательские и экспертные оценки.

| Столбец | Описание |
|---------|----------|
| `Name` | название игры |
| `Platform` | название платформы |
| `Year of Release` | год выпуска игры |
| `Genre` | жанр игры |
| `NA sales` | продажи в Северной Америке (млн копий) |
| `EU sales` | продажи в Европе (млн копий) |
| `JP sales` | продажи в Японии (млн копий) |
| `Other sales` | продажи в других странах (млн копий) |
| `Critic Score` | оценка критиков (от 0 до 100) |
| `User Score` | оценка пользователей (от 0 до 10) |
| `Rating` | рейтинг ESRB (возрастная категория) |

---

## Содержимое проекта

[1. Знакомство с данными и их описание](#1)

[2. Проверка ошибок в данных и их предобработка](#2)
   - 2.1. Названия столбцов
   - 2.2. Типы данных
   - 2.3. Наличие пропусков в данных
   - 2.4. Явные и неявные дубликаты

[3. Фильтрация данных (2000–2013)](#3)

[4. Категоризация данных](#4)

[5. Итоговый вывод](#5)

---

## 1. Загрузка данных и знакомство с ними <a id='1'></a>

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('new_games.csv')

In [3]:
df.head()

,Name,Platform,Year of Release,Genre,NA sales,EU sales,JP sales,Other sales,Critic Score,User Score,Rating
0,Wii Sports,Wii,2006.0,Sports,41.36,28.96,3.77,8.45,76.0,8,E
1,Super Mario Bros.,NES,1985.0,Platform,29.08,3.58,6.81,0.77,NaN,NaN,NaN
2,Mario Kart Wii,Wii,2008.0,Racing,15.68,12.76,3.79,3.29,82.0,8.3,E
3,Wii Sports Resort,Wii,2009.0,Sports,15.61,10.93,3.28,2.95,80.0,8,E
4,Pokemon Red/Pokemon Blue,GB,1996.0,Role-Playing,11.27,8.89,10.22,1.00,NaN,NaN,NaN


In [4]:
initial_rows = len(df)
print(f"Исходное количество строк: {initial_rows}")

Исходное количество строк: 16956


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 16956 entries, 0 to 16955
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Name             16954 non-null  str    
 1   Platform         16956 non-null  str    
 2   Year of Release  16681 non-null  float64
 3   Genre            16954 non-null  str    
 4   NA sales         16956 non-null  float64
 5   EU sales         16956 non-null  str    
 6   JP sales         16956 non-null  str    
 7   Other sales      16956 non-null  float64
 8   Critic Score     8242 non-null   float64
 9   User Score       10152 non-null  str    
 10  Rating           10085 non-null  str    
dtypes: float64(4), str(7)
memory usage: 1.4 MB


### Вывод по структуре данных

- **Объем данных:** 16 956 строк, 11 столбцов, объем памяти 1.4 МБ
- **Структура** соответствует описанию, все необходимые поля присутствуют

**На что обратить внимание при предобработке:**

1. **Пропуски** — есть в столбцах:
   - `Name`, `Genre` — единичные пропуски
   - `Year of Release` — 275 пропусков
   - `Critic Score` — более 50% пропусков (8242 из 16956)
   - `User Score`, `Rating` — также значительные пропуски (>10%)

2. **Несоответствие типов данных:**
   - `Year of Release` — тип `float64`, лучше перевести в `int`
   - `EU sales`, `JP sales`, `User Score` — тип `str`, нужно перевести в `float64`

3. **Названия столбцов** — в неудобном формате (с пробелами), перед анализом приведу к единому стилю `snake_case`

---

## 2.  Проверка ошибок в данных и их предобработка <a id='2'></a>


### 2.1. Названия, или метки, столбцов датафрейма

In [6]:
print("Текущие названия столбцов:")
print(df.columns)

Текущие названия столбцов:
Index(['Name', 'Platform', 'Year of Release', 'Genre', 'NA sales', 'EU sales',
       'JP sales', 'Other sales', 'Critic Score', 'User Score', 'Rating'],
      dtype='str')


In [7]:
df.columns = (df.columns
              .str.strip()
              .str.lower()
              .str.replace(' ', '_'))

In [8]:
print("\nНовые названия столбцов (snake_case):")
print(df.columns)


Новые названия столбцов (snake_case):
Index(['name', 'platform', 'year_of_release', 'genre', 'na_sales', 'eu_sales',
       'jp_sales', 'other_sales', 'critic_score', 'user_score', 'rating'],
      dtype='str')


**Вывод:** Названия столбцов приведены к единому стилю `snake_case` — все буквы нижнего регистра, пробелы заменены на подчёркивания.

### 2.2. Типы данных

In [9]:
df['year_of_release'] = pd.to_numeric(df['year_of_release'], errors = 'raise', downcast = 'integer')

In [10]:
errors = (df['eu_sales'] == 'unknown').sum()
print(f'Количество "unknown" в столбце "eu_sales": {errors}')

Количество "unknown" в столбце "eu_sales": 6


In [11]:
df['eu_sales'] = pd.to_numeric(df['eu_sales'], errors = 'coerce', downcast = 'float')

In [12]:
errors = (df['jp_sales'] == 'unknown').sum()
print(f'Количество "unknown" в столбце "jp_sales": {errors}')

Количество "unknown" в столбце "jp_sales": 4


In [13]:
df['jp_sales'] = pd.to_numeric(df['jp_sales'], errors = 'coerce', downcast = 'float')

In [14]:
errors = (df['user_score'] == 'tbd').sum()
print(f'Количество "tbd" в столбце "user_score": {errors}')

Количество "tbd" в столбце "user_score": 2464


In [15]:
df['user_score'] = pd.to_numeric(df['user_score'], errors = 'coerce', downcast = 'float')

In [16]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 16956 entries, 0 to 16955
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   name             16954 non-null  str    
 1   platform         16956 non-null  str    
 2   year_of_release  16681 non-null  float64
 3   genre            16954 non-null  str    
 4   na_sales         16956 non-null  float64
 5   eu_sales         16950 non-null  float32
 6   jp_sales         16952 non-null  float32
 7   other_sales      16956 non-null  float64
 8   critic_score     8242 non-null   float64
 9   user_score       7688 non-null   float32
 10  rating           10085 non-null  str    
dtypes: float32(3), float64(4), str(4)
memory usage: 1.2 MB


**Вывод по типам данных:**

После преобразования типов данных:

- `year_of_release` — приведён к числовому типу (`float64`)
- `eu_sales`, `jp_sales`, `user_score` — преобразованы в `float32` (некорректные строки заменены на `NaN`)

**Потери данных при преобразовании:**

| Столбец | Причина потери | Количество |
|---------|----------------|------------|
| `eu_sales` | значение "unknown" | 6 |
| `jp_sales` | значение "unknown" | 4 |
| `user_score` | значение "tbd" (to be determined) | 2464 |

Эти значения не могли быть преобразованы в числа и были заменены на пропуски, что является корректным подходом при работе с такими данными.


### 2.3. Наличие пропусков в данных

In [17]:
missing_values = pd.DataFrame({
    'Количество пропусков': df.isna().sum(),
    'Процент пропусков (%)': round((df.isna().sum() / len(df) * 100), 2)
})

In [18]:
print(missing_values)

                 Количество пропусков  Процент пропусков (%)
name                                2                   0.01
platform                            0                   0.00
year_of_release                   275                   1.62
genre                               2                   0.01
na_sales                            0                   0.00
eu_sales                            6                   0.04
jp_sales                            4                   0.02
other_sales                         0                   0.00
critic_score                     8714                  51.39
user_score                       9268                  54.66
rating                           6871                  40.52


**Промежуточный вывод по пропускам:**

| Столбец | Количество пропусков | % | Причина / Действие |
|---------|----------------------|---|--------------------|
| `name` | 2 | <0.01% | удаляю |
| `genre` | 2 | <0.01% | удаляю |
| `year_of_release` | 275 | 1.62% | не удаляю — данные не пройдут фильтрацию по 2000–2013 |
| `eu_sales` | 6 | 0.04% | заменяю на среднее по `platform`+`year_of_release` |
| `jp_sales` | 4 | 0.02% | заменяю на среднее по `platform`+`year_of_release` |
| `critic_score` | 8714 | 51.39% | оставляю как `NaN` (позже категоризация) |
| `user_score` | 9268 | 54.66% | оставляю как `NaN` (позже категоризация) |
| `rating` | 6871 | 40.52% | не используется в анализе |

In [19]:
df = df.dropna(subset=['name', 'genre'])

In [20]:
df['eu_sales'] = df['eu_sales'].fillna(df.groupby(['platform','year_of_release'])['eu_sales'].transform('mean'))
df['jp_sales'] = df['jp_sales'].fillna(df.groupby(['platform','year_of_release'])['jp_sales'].transform('mean'))
df.info()

<class 'pandas.DataFrame'>
Index: 16954 entries, 0 to 16955
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   name             16954 non-null  str    
 1   platform         16954 non-null  str    
 2   year_of_release  16679 non-null  float64
 3   genre            16954 non-null  str    
 4   na_sales         16954 non-null  float64
 5   eu_sales         16954 non-null  float32
 6   jp_sales         16954 non-null  float32
 7   other_sales      16954 non-null  float64
 8   critic_score     8242 non-null   float64
 9   user_score       7688 non-null   float32
 10  rating           10085 non-null  str    
dtypes: float32(3), float64(4), str(4)
memory usage: 1.4 MB


In [21]:
df[['eu_sales', 'jp_sales']].isna().sum()

eu_sales    0
jp_sales    0
dtype: int64

**Вывод по обработке пропусков:**

- Пропуски в `name` и `genre` удалены (2 строки)
- Пропуски в `eu_sales` (6 записей) и `jp_sales` (4 записи) заполнены средними значениями по группам `platform` + `year_of_release`
- Пропуски в `year_of_release` оставлены — строки `null` не пройдут фильтрацию по годам
- Пропуски в `critic_score`, `user_score` и `rating` оставлены как есть (не используются в категоризации или не требуют замены)


### 2.4. Явные и неявные дубликаты в данных

In [22]:
df['genre'].unique()

<StringArray>
[      'Sports',     'Platform',       'Racing', 'Role-Playing',
       'Puzzle',         'Misc',      'Shooter',   'Simulation',
       'Action',     'Fighting',    'Adventure',     'Strategy',
         'MISC', 'ROLE-PLAYING',       'RACING',       'ACTION',
      'SHOOTER',     'FIGHTING',       'SPORTS',     'PLATFORM',
    'ADVENTURE',   'SIMULATION',       'PUZZLE',     'STRATEGY']
Length: 24, dtype: str

In [23]:
df['platform'].unique()

<StringArray>
[ 'Wii',  'NES',   'GB',   'DS', 'X360',  'PS3',  'PS2', 'SNES',  'GBA',
  'PS4',  '3DS',  'N64',   'PS',   'XB',   'PC', '2600',  'PSP', 'XOne',
 'WiiU',   'GC',  'GEN',   'DC',  'PSV',  'SAT',  'SCD',   'WS',   'NG',
 'TG16',  '3DO',   'GG', 'PCFX']
Length: 31, dtype: str

In [24]:
df['rating'].unique()

<StringArray>
['E', nan, 'M', 'T', 'E10+', 'K-A', 'AO', 'EC', 'RP']
Length: 9, dtype: str

In [25]:
df['year_of_release'].unique()

array([2006., 1985., 2008., 2009., 1996., 1989., 1984., 2005., 1999.,
       2007., 2010., 2013., 2004., 1990., 1988., 2002., 2001., 2011.,
       1998., 2015., 2012., 2014., 1992., 1997., 1993., 1994., 1982.,
       2016., 2003., 1986., 2000.,   nan, 1995., 1991., 1981., 1987.,
       1980., 1983.])

In [26]:
df['genre'] = df['genre'].str.lower()
df['platform'] = df['platform'].str.lower()
df['rating'] = df['rating'].str.upper()

In [27]:
df['genre'].unique()

<StringArray>
[      'sports',     'platform',       'racing', 'role-playing',
       'puzzle',         'misc',      'shooter',   'simulation',
       'action',     'fighting',    'adventure',     'strategy']
Length: 12, dtype: str

In [28]:
df.duplicated(subset=['name', 'platform', 'year_of_release', 'genre']).sum()

np.int64(242)

In [29]:
df[df.duplicated(subset=['name', 'platform', 'year_of_release', 'genre'], keep=False)].sort_values(['name', 'platform', 'year_of_release', 'genre'])

,name,platform,year_of_release,genre,na_sales,eu_sales,jp_sales,other_sales,critic_score,user_score,rating
15191,Beyblade Burst,3ds,2016.0,role-playing,0.00,0.00,0.03,0.00,NaN,NaN,NaN
15192,Beyblade Burst,3ds,2016.0,role-playing,0.00,0.00,0.03,0.00,NaN,NaN,NaN
15301,11eyes: CrossOver,x360,2009.0,adventure,0.00,0.00,0.02,0.00,NaN,NaN,NaN
15302,11eyes: CrossOver,x360,2009.0,adventure,0.00,0.00,0.02,0.00,NaN,NaN,NaN
4860,18 Wheeler: American Pro Trucker,ps2,2001.0,racing,0.20,0.15,0.00,0.05,61.0,5.7,E
...,...,...,...,...,...,...,...,...,...,...,...
2909,Yu-Gi-Oh! The Falsebound Kingdom,gc,2002.0,strategy,0.49,0.13,0.07,0.02,NaN,NaN,NaN
6695,Zoo Resort 3D,3ds,2011.0,simulation,0.11,0.09,0.03,0.02,NaN,NaN,E
6696,Zoo Resort 3D,3ds,2011.0,simulation,0.11,0.09,0.03,0.02,NaN,NaN,E
8156,Zumba Fitness Rush,x360,2012.0,sports,0.00,0.16,0.00,0.02,73.0,6.2,E10+


In [30]:
df = df.drop_duplicates(subset=['name', 'platform', 'year_of_release', 'genre'])

In [31]:
print(df.duplicated().sum())

0


In [32]:
df = df.drop_duplicates()
print(df.duplicated().sum())

0


**Промежуточный вывод по дубликатам:**

- В столбце `genre` обнаружены неявные дубликаты из-за разного регистра написания. После приведения к нижнему регистру количество уникальных жанров сократилось с 24 до 12.
- Выявлены **241 явный дубликат** (полностью совпадающие строки по ключевым полям). Они удалены с помощью `drop_duplicates()`.
- После удаления дубликатов полных совпадений строк не осталось

In [33]:
final_rows = len(df)
print(f"Финальное количество строк: {final_rows}")

Финальное количество строк: 16712


In [34]:
rows_removed = initial_rows - final_rows
percent_removed = (rows_removed / initial_rows) * 100

print(f"Удалено строк: {rows_removed}")
print(f"Удалено {percent_removed:.2f}% от исходного количества")

Удалено строк: 244
Удалено 1.44% от исходного количества


**Количество удалённых строк:**

Было удалено:
- 2 строки с пропусками в `name` и `genre`
- 241 строка явных дубликатов

Исходное количество строк: 16 956  
Финальное количество строк: 16 712  

**Удалено 244 строки (1.44% от исходного количества).**


**Общий вывод по предобработке данных:**

В результате выполненных действий:

| Этап | Действие | Результат |
|------|----------|-----------|
| Пропуски | Удалены строки с `name` и `genre` | -2 строки |
| Пропуски | Заполнены `eu_sales`, `jp_sales` средними по платформе и году | 0 пропусков |
| Пропуски | `critic_score`, `user_score` оставлены как `NaN` | для категоризации |
| Нормализация | `genre`, `platform` → нижний регистр; `rating` → верхний | устранены неявные дубликаты |
| Дубликаты | Удалены явные дубликаты (241 строка) | очистка данных |

**Итог:** из исходных 16 956 строк удалено 244 строки (1.44%).  
Финальное количество строк — **16 712**. Данные готовы к дальнейшему анализу (фильтрация по годам, категоризация оценок).

---

## 3. Фильтрация данных <a id='3'></a>

In [35]:
df_actual = df[(df['year_of_release'] >= 2000) & (df['year_of_release'] <= 2013)].copy()

In [36]:
print(len(df_actual))
print(df_actual['year_of_release'].min())
print(df_actual['year_of_release'].max())
print(df_actual['year_of_release'].isna().sum())

12780
2000.0
2013.0
0


**Вывод по фильтрации:**

- Исходное количество строк после предобработки: **16 712**
- После фильтрации по периоду 2000–2013 осталось: **12 780 строк**
- Минимальный год: **2000**, максимальный: **2013**
- Пропусков в столбце `year_of_release` нет (благодаря фильтрации)

**Потеряно строк:** 16 712 − 12 780 = **3 932 строки** (игры, выпущенные до 2000 года или с пропущенным годом выпуска).

---

## 4. Категоризация данных <a id='4'></a>
На основе пользовательских и экспертных оценок разделим игры на три категории: высокая, средняя, низкая. Пропуски заполним значением `'нет оценки'`.

In [37]:
def categorize_user(score):
    if pd.isna(score):
        return None
    elif score >= 8:
        return 'высокая оценка'
    elif score >= 3:
        return 'средняя оценка'
    elif score >= 0:
        return 'низкая оценка'

In [38]:
df_actual['user_category'] = df_actual['user_score'].apply(categorize_user)

In [39]:
df_actual['user_category'] = df_actual['user_category'].fillna('нет оценки')

In [40]:
def categorize_critic(score):
    if pd.isna(score):
        return None
    elif score >= 80:
        return 'высокая оценка'
    elif score >= 30:
        return 'средняя оценка'
    elif score >= 0:
        return 'низкая оценка'

In [41]:
df_actual['critic_category'] = df_actual['critic_score'].apply(categorize_critic)

In [42]:
df_actual['critic_category'] = df_actual['critic_category'].fillna('нет оценки')

In [43]:
df_actual['critic_category'].value_counts()

critic_category
нет оценки        5612
средняя оценка    5422
высокая оценка    1691
низкая оценка       55
Name: count, dtype: int64

In [44]:
df_actual['user_category'].value_counts()

user_category
нет оценки        6298
средняя оценка    4080
высокая оценка    2286
низкая оценка      116
Name: count, dtype: int64

**Вывод по категоризации оценок:**

**Оценки критиков (`critic_score`):**

| Категория | Количество игр |
|-----------|----------------|
| нет оценки | 5 612 |
| средняя оценка | 5 422 |
| высокая оценка | 1 691 |
| низкая оценка | 55 |

**Оценки пользователей (`user_score`):**

| Категория | Количество игр |
|-----------|----------------|
| нет оценки | 6 298 |
| средняя оценка | 4 080 |
| высокая оценка | 2 286 |
| низкая оценка | 116 |

**Наблюдения:**
- Значительная часть игр не имеет оценок (особенно у пользователей — 6298)
- Средние оценки преобладают у обеих групп
- Низких оценок крайне мало (55 у критиков, 116 у пользователей)
- Высоких оценок больше у пользователей (2286 против 1691 у критиков)


In [45]:
platform_counts = df_actual['platform'].value_counts()

In [46]:
top_platforms = platform_counts.head(7)

In [47]:
print(top_platforms)

platform
ps2     2127
ds      2120
wii     1275
psp     1180
x360    1121
ps3     1086
gba      811
Name: count, dtype: int64


**Топ-7 платформ:**

| Платформа | Количество игр |
|-----------|----------------|
| PS2 | 2 127 |
| DS | 2 120 |
| Wii | 1 275 |
| PSP | 1 180 |
| X360 | 1 121 |
| PS3 | 1 086 |
| GBA | 811 |

**Вывод:** Лидерами по количеству выпущенных игр за период 2000–2013 являются **PS2** и **DS** (более 2000 игр каждая), что соответствует пику популярности этих платформ в те годы.

## 5. Итоговый вывод

В ходе выполнения проекта выполнена полная предобработка исторических данных о продажах видеоигр.

### Исходные данные
- **Количество строк:** 16 956
- **Количество столбцов:** 11
- **Период:** 1980–2016 гг.

### Предобработка данных
| Действие | Результат |
|----------|-----------|
| Приведение названий столбцов к `snake_case` | все столбцы в едином формате |
| Преобразование типов данных | `year_of_release` → `float64`, `eu_sales`, `jp_sales`, `user_score` → `float32` |
| Замена `'unknown'` и `'tbd'` на `NaN` | 6 (`eu_sales`), 4 (`jp_sales`), 2464 (`user_score`) |
| Заполнение пропусков в `eu_sales`, `jp_sales` | средними значениями по `platform` и `year_of_release` |
| Удаление строк с пропусками в `name`, `genre` | -2 строки |
| Нормализация регистра | `genre`, `platform` → нижний, `rating` → верхний |
| Удаление явных дубликатов | -242 строки |
| Фильтрация по периоду 2000–2013 | осталось **12 780 строк** |

### Новые поля
- `user_category` — категория на основе `user_score` (высокая / средняя / низкая / нет оценки)
- `critic_category` — категория на основе `critic_score`

**Распределение по категориям:**

| Категория | critic_category | user_category |
|-----------|----------------|---------------|
| нет оценки | 5 612 | 6 298 |
| высокая оценка | 1 691 | 2 286 |
| средняя оценка | 5 422 | 4 080 |
| низкая оценка | 55 | 116 |

### Топ-7 платформ (2000–2013)

| Платформа | Количество игр |
|-----------|----------------|
| PS2 | 2 127 |
| DS | 2 120 |
| Wii | 1 275 |
| PSP | 1 180 |
| X360 | 1 121 |
| PS3 | 1 086 |
| GBA | 811 |

### Общий вывод

Данные успешно очищены, подготовлены и структурированы. Все этапы задания выполнены. Полученный срез (2000–2013) и новые категориальные поля пригодны для дальнейшего анализа продаж, динамики по годам и связи оценок с популярностью жанров и платформ.